In [1]:
from llama_index.core import Settings
# from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
import os

from dotenv import load_dotenv
load_dotenv()
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")
Settings.llm = OpenAI(model="gpt-3.5-turbo", temperature=0)

/Users/rahultiwari/Documents/02_Freelancing/coding_ninja_fresh/dummy-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pip install llama-index-vector-stores-pinecone

  Using cached llama_index_vector_stores_pinecone-0.8.0-py3-none-any.whl.metadata (425 bytes)
Using cached llama_index_vector_stores_pinecone-0.8.0-py3-none-any.whl (8.0 kB)

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# seeting up pinecone
from pinecone import Pinecone, ServerlessSpec
from llama_index.vector_stores.pinecone import PineconeVectorStore
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import SimpleDirectoryReader


In [4]:
pc = Pinecone()

In [11]:
existing= pc.list_indexes()

index_name = []

for index in existing:
    index_name.append(index.name)
    
print(index_name)

['documind-product', 'documind-hr', 'codesage', 'documind-finance', 'semantic-search-fast']


In [12]:
indexes_to_delete = ["documind-hr", "documind-product", "documind-finance"]

for idx_name in indexes_to_delete:
    if idx_name in index_name:
        pc.delete_index(idx_name)
        print(f"Deleted index: {idx_name}")
    else:
        print(f"Index {idx_name} not found.")


Deleted index: documind-hr
Deleted index: documind-product
Deleted index: documind-finance


In [13]:
INDEX_CONFIGS = {
    "documind-hr":      "hr_policy_manual.pdf",
    "documind-product": "novacrm_product_manual.pdf",
    "documind-finance": "annual_financial_report_2024.pdf"
}

In [15]:
def get_or_create_index(index_name,pdf_path):
    existing = [i.name for i in pc.list_indexes()]
    if index_name not in existing:
        print(f"creating index: {index_name}")
        pc.create_index(
            name=index_name,
            dimension=1536,       # must match embedding model
            metric="cosine",       # cosine similarity for text
            spec=ServerlessSpec(cloud="aws", region="us-east-1")
        )

    docs = SimpleDirectoryReader(
        input_files=[pdf_path]
    ).load_data()
    nodes = SentenceSplitter(chunk_size=512, chunk_overlap=64).get_nodes_from_documents(docs)
    # store it in Pinecone
    vs = PineconeVectorStore(pinecone_index=pc.Index(index_name))
    storage_context = StorageContext.from_defaults(vector_store=vs)
    index = VectorStoreIndex(nodes, storage_context=storage_context)
    print(f"index created: {index_name}")
    return index

In [16]:
hr_index      = get_or_create_index("documind-hr",      "data_test/hr_policy_manual.pdf")
product_index = get_or_create_index("documind-product", "data_test/novacrm_product_manual.pdf")
finance_index = get_or_create_index("documind-finance", "data_test/annual_financial_report_2024.pdf")


creating index: documind-hr


2026-06-07 11:39:51,713 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
Upserted vectors: 100%|██████████| 5/5 [00:02<00:00,  2.37it/s]


index created: documind-hr
creating index: documind-product


2026-06-07 11:40:05,482 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
Upserted vectors: 100%|██████████| 6/6 [00:02<00:00,  2.57it/s]


index created: documind-product
creating index: documind-finance


2026-06-07 11:40:18,883 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
Upserted vectors: 100%|██████████| 6/6 [00:02<00:00,  2.49it/s]

index created: documind-finance


In [17]:
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector

In [18]:
from llama_index.core import PromptTemplate

PROMPT = PromptTemplate("""
You are DocuMind — NovaTech's knowledge assistant.
RULES:
- Use ONLY the CONTEXT. No outside knowledge.
- If not in CONTEXT: "Not found in provided documents."
- Max 4 sentences. Add source tag [HR], [PRODUCT], or [FINANCE].
CONTEXT: {context_str}
QUESTION: {query_str}
ANSWER:
""")


In [19]:
hr_tools = QueryEngineTool(
    query_engine=hr_index.as_query_engine(
        text_qa_template=PROMPT,
        similarity_top_k=3,
    ),
    metadata=ToolMetadata(
        name="HR Policy Manual",
        description=(
            "Use for HR and employee policy questions: leave policies "
            "(annual, sick, maternity, paternity, casual, bereavement), "
            "notice period, probation, salary structure, EPF, appraisal, "
            "code of conduct, disciplinary action, WFH policy."
        )
    )
    
)


finance_tool = QueryEngineTool(
    query_engine=finance_index.as_query_engine(
        text_qa_template=PROMPT, similarity_top_k=3
    ),
    metadata=ToolMetadata(
        name="financial_report",
        description=(
            "Use for financial and business performance questions: "
            "revenue, profit, EBITDA, growth, Q1/Q2/Q3/Q4 performance, "
            "risk factors, strategic outlook, international expansion, "
            "employee count, attrition, segment analysis."
        )
    )
)




product_tool = QueryEngineTool(
    query_engine=product_index.as_query_engine(
        text_qa_template=PROMPT, similarity_top_k=3
    ),
    metadata=ToolMetadata(
        name="product_manual",
        description=(
            "Use for NovaCRM product questions: features, modules, "
            "sales module, customer support, marketing automation, "
            "reporting analytics, integrations, API, pricing tiers, "
            "technical specifications, SLA."
        )
    )
)


In [21]:
router = RouterQueryEngine.from_defaults(
    query_engine_tools=[hr_tools, product_tool, finance_tool],
    selector=LLMSingleSelector.from_defaults(),
    verbose=True
   

)

In [ ]:
# What is the maternity leave policy?

In [22]:
router.query("What is the maternity leave policy?")

2026-06-07 11:52:10,631 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-07 11:52:10,648 - INFO - Selecting query engine 0: Choice 1 is most relevant as it specifically mentions maternity leave policy among other HR and employee policy questions..


Selecting query engine 0: Choice 1 is most relevant as it specifically mentions maternity leave policy among other HR and employee policy questions..


2026-06-07 11:52:11,131 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-07 11:52:14,196 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Response(response='Female employees are entitled to 26 weeks (182 days) of paid Maternity Leave for the first two children. For the third child onwards, the entitlement is 12 weeks. Applicable only for employees who have worked at least 80 days in the preceding 12 months. [HR]', source_nodes=[NodeWithScore(node=TextNode(id_='4e3b062b-4827-4c0d-9a0d-6b8d131dbe28', embedding=[-0.00338554382, 0.0553894043, 0.100952148, 0.00155067444, -0.0216674805, -0.00517654419, 0.0108718872, -0.00366592407, 0.0333557129, -0.0017786026, 0.0366516113, -0.0236969, 0.0346679688, -0.0198516846, 0.0405578613, 0.0143432617, 0.0419006348, -0.0285186768, -0.0236358643, 0.0654296875, 0.0166168213, 0.0800170898, -0.0073890686, 0.0144500732, 0.0036277771, 0.00776290894, -0.013961792, -0.00655365, 0.0171051025, -0.0863037109, 0.0203552246, -0.0157165527, -0.0245666504, 0.0382995605, 0.00579834, 0.0316467285, 0.00268173218, 0.0127334595, 0.00984954834, 0.00270652771, -0.0286712646, -0.0117645264, 0.022354126, -0.014

In [23]:
test_queries = [
    ("What is the maternity leave policy?",        "→ should route to hr_policy"),
    ("What does NovaCRM's Sales Module do?",       "→ should route to product_manual"),
    ("Why did Q3 FY24 revenue grow so strongly?",  "→ should route to financial_report"),
]

print("RouterQueryEngine — Routing Demo")
print("="*55)
for q, expected in test_queries:
    print(f"\nQ: {q}")
    print(f"Expected: {expected}")
    r = router.query(q)
    print(f"A: {r}")
    print("-"*40)

RouterQueryEngine — Routing Demo

Q: What is the maternity leave policy?
Expected: → should route to hr_policy


2026-06-07 11:52:54,596 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-07 11:52:54,602 - INFO - Selecting query engine 0: Choice 1 is most relevant as it specifically mentions maternity leave policy along with other HR and employee policy questions..


Selecting query engine 0: Choice 1 is most relevant as it specifically mentions maternity leave policy along with other HR and employee policy questions..


2026-06-07 11:52:55,048 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-07 11:52:57,473 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


A: Female employees are entitled to 26 weeks (182 days) of paid Maternity Leave for the first two children. For the third child onwards, the entitlement is 12 weeks. Applicable only for employees who have worked at least 80 days in the preceding 12 months. [HR]
----------------------------------------

Q: What does NovaCRM's Sales Module do?
Expected: → should route to product_manual


...

A: The Sales Module manages the complete B2B and B2C sales lifecycle from lead capture to deal closure. It supports both direct sales teams and channel partner networks. Leads are captured from various sources and automatically scored based on behavioral and firmographic signals. The module also includes an opportunity pipeline that supports up to 10 customizable deal stages. [PRODUCT]
----------------------------------------

Q: Why did Q3 FY24 revenue grow so strongly?
Expected: → should route to financial_report


2026-06-07 11:53:03,214 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-06-07 11:53:03,220 - INFO - Selecting query engine 2: Choice 3 is most relevant as it pertains to financial and business performance questions, including revenue growth, Q3 performance, and strategic outlook..


Selecting query engine 2: Choice 3 is most relevant as it pertains to financial and business performance questions, including revenue growth, Q3 performance, and strategic outlook..


2026-06-07 11:53:03,471 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-06-07 11:53:06,529 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


A: Q3 FY24 revenue grew strongly due to the festive season WhatsApp campaign spike for e-commerce clients, year-end budget utilisation by enterprise IT clients, and successful expansion of Managed Services to 3 new banking clients. International revenue from UAE and Singapore also contributed to the growth. [FINANCE]
----------------------------------------
